# Probe: kualitas Bahasa Indonesia 3 model NIM (alat keputusan)

Tujuan: **membandingkan langsung** kualitas Bahasa Indonesia dari tiga kandidat generator Module 06, lalu memutuskan secara **empiris** (bukan asumsi).

| Model | Lineage | Catatan |
|-------|---------|---------|
| `nvidia/nvidia-nemotron-nano-9b-v2` | **NVIDIA-native** (Mamba-Transformer) | bahasa resmi: EN+DE/FR/IT/ES/JA — ID **tidak** terdaftar |
| `nvidia/llama-3.3-nemotron-super-49b-v1.5` | Llama-based, NVIDIA-tuned | multilingual Llama 3.3 (ID kuat) |
| `meta/llama-3.3-70b-instruct` | Meta | referensi ID terbaik (dipakai M05) |

> Baca jawaban tiap model untuk 3 prompt, nilai **mana yang paling natural & tepat** dalam Bahasa Indonesia.

In [ ]:
!pip -q install openai
import os
if not os.path.exists("nvidia_utils.py"):
    !wget -q https://raw.githubusercontent.com/chmdznr/navasena-gen-ml-course/module06-rework/06_nvidia_ecosystem/tools/nvidia_utils.py
from google.colab import userdata
os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")
print("Key termuat:", bool(os.environ.get("NVIDIA_API_KEY")))

In [ ]:
import re
from nvidia_utils import nim_client
client = nim_client()

MODELS = {
    "Nemotron-Nano-9B-v2 (NVIDIA-native)": "nvidia/nvidia-nemotron-nano-9b-v2",
    "Llama-Nemotron-Super-49B (NVIDIA-tuned)": "nvidia/llama-3.3-nemotron-super-49b-v1.5",
    "Llama-3.3-70B (Meta)": "meta/llama-3.3-70b-instruct",
}
NEMOTRON = {"nvidia/nvidia-nemotron-nano-9b-v2", "nvidia/llama-3.3-nemotron-super-49b-v1.5"}

def ask(model, prompt):
    msgs = []
    if model in NEMOTRON:
        msgs.append({"role": "system", "content": "/no_think"})   # matikan reasoning
    msgs.append({"role": "user", "content": prompt})
    out = client.chat.completions.create(model=model, messages=msgs, temperature=0).choices[0].message.content
    return re.sub(r"<think>.*?</think>", "", out, flags=re.DOTALL).strip()   # buang sisa blok think

In [ ]:
PROMPTS = [
    ("Formal/edukasi", "Jelaskan apa itu retrieval-augmented generation (RAG) kepada lulusan SMA dalam 2 kalimat."),
    ("Layanan pelanggan", "Pelanggan bertanya: 'Kok pesanan saya belum sampai ya, padahal sudah 3 hari?' Jawab dengan sopan dan membantu."),
    ("Santai/idiomatik", "Beri satu tips singkat memilih GPU untuk belajar AI, pakai bahasa santai khas anak muda Indonesia."),
]
for judul, p in PROMPTS:
    print("=" * 90)
    print(f"PROMPT [{judul}]: {p}")
    print("=" * 90)
    for nama, model in MODELS.items():
        try:
            jawab = ask(model, p)
        except Exception as e:
            jawab = f"(error: {str(e)[:120]})"
        print(f"\n--- {nama} ---\n{jawab}\n")
    print()

## Keputusan

Baca ketiga jawaban untuk tiap prompt. Pertanyaan kunci:
- Apakah **Nemotron-Nano-9B-v2** (NVIDIA-native) menjawab Bahasa Indonesia dengan **natural & benar**, atau terasa kaku/keliru?
- Jika **native-nya bagus** → pakai itu untuk Module 06 (paling NVIDIA + terkini).
- Jika **kaku/keliru** → pakai **Llama-Nemotron-Super-49B** (NVIDIA-tuned, ID kuat) atau **Llama-3.3-70B**.

Beri tahu hasilnya, nanti aku set model Module 06 sesuai bukti ini.